# Load modules and define functions
## Imports

In [1]:
import os
import seaborn as sns
import scanpy as sc
import scipy.sparse as sp

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

data_dir         = "/Users/wsun/research/CAT/data/"
results_dir      = "classification_result/"

cell_type = "CD4"

## Read in scRNA-seq data

In [2]:
adata = sc.read_h5ad(data_dir + cell_type + "_combined_filtered.h5ad")

print("Count Data:")
print(adata.X.shape)
print(adata.X[:5,:4])
print()
print("Meta Data:")
print(adata.obs.shape)
pd.set_option('display.max_columns', None)  # show all columns
print(adata.obs[:5])
print()

Count Data:
(390785, 6241)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (5, 4)>
  Coords	Values
  (2, 1)	1.0
  (4, 3)	1.0

Meta Data:
(390785, 44)
                                           study2  n_genes_by_counts  \
AGTGGGATCTGACCTC.58-THCA-Zheng_2021    Zheng_2021               1346   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024    Chen_2024               1029   
CRC04-B-II_ATTTCTGCAGTAAGCG-Chen_2024   Chen_2024               1236   
CGCTTCAAGGCAAAGA-1_PEM15C5-Chow_2023    Chow_2023                971   
P47-TCAGCTCGTTCAGCGC-1-Liu_2025          Liu_2025               1041   

                                       total_counts         TRB_cdr3  \
AGTGGGATCTGACCTC.58-THCA-Zheng_2021          3215.0  CA*MRGFIMATPSVR   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024         2930.0    CAAAATNNNEQFF   
CRC04-B-II_ATTTCTGCAGTAAGCG-Chen_2024        3943.0     CAAAGAGTEAFF   
CGCTTCAAGGCAAAGA-1_PEM15C5-Chow_2023         2697.0   CAAAGGPKSGELFF   
P47-TCAGCTCGTTC

## Read in NN prediction results

In [3]:
# list folders starting with "CD8"
folders = [
    f for f in os.listdir(results_dir)
    if f.startswith(cell_type) and os.path.isdir(os.path.join(results_dir, f))
]

print(f"{cell_type} folders:")
print(folders)

CD4 folders:
['CD4_Zheng_2021_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20', 'CD4_Liu_2025_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20', 'CD4_Chen_2024_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20', 'CD4_Liu_2022_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20']


In [4]:
file_path = os.path.join(results_dir, folders[0], "predictions_test_data.txt")

df = pd.read_csv(file_path, sep="\t", index_col=0)
print("Predictions DataFrame:")
print(df.head())

Predictions DataFrame:
                                       Prediction  True Label
AGTGGGATCTGACCTC.58-THCA-Zheng_2021  9.780438e-07           0
CTTTGCGAGTCATGCT.82-UCEC-Zheng_2021  8.348551e-04           0
CTCGAGGTCTACTATC.14-ESCA-Zheng_2021  6.481948e-02           0
CTGCTGTTCAGTGCAT.14-ESCA-Zheng_2021  7.764926e-04           0
TCATTACAGAGGACGG.14-ESCA-Zheng_2021  1.170999e-02           0


In [5]:

# list folders starting with "CD8" or "CD4"
folders = [
    f for f in os.listdir(results_dir)
    if f.startswith(cell_type) and os.path.isdir(os.path.join(results_dir, f))
]

frames = []

for folder in folders:
    file_path = os.path.join(results_dir, folder, "predictions_test_data.txt")
    if not os.path.exists(file_path):
        print(f"❓ File not found: {file_path}")
        continue

    df = pd.read_csv(file_path, sep="\t", index_col=0)

    # check for any IDs in df not present in adata.obs
    missing_in_obs = df.index.difference(adata.obs.index)
    if len(missing_in_obs) > 0:
        print(f"⚠️  Skipping {folder}: {len(missing_in_obs)} cell IDs not in adata.obs (showing up to 10):")
        print(list(missing_in_obs[:10]))
        continue

    # (optional) drop any rows not in obs, though above check ensures none
    df = df.loc[df.index.intersection(adata.obs.index)]

    frames.append(df)

if not frames:
    print("⚠️  No valid prediction files were loaded; nothing to merge.")
else:
    # row-wise bind (keep cell IDs as index)
    combined = pd.concat(frames, axis=0)

    # warn on duplicate cell IDs across folders
    dup_mask = combined.index.duplicated(keep=False)
    n_dups = dup_mask.sum()
    if n_dups > 0:
        dup_ids = combined.index[dup_mask].unique()
        print(f"⚠️  {n_dups} duplicate rows detected across folders "
              f"({len(dup_ids)} unique cell IDs). Keeping the first occurrence per cell ID.")
        # keep first occurrence per cell ID
        combined = combined[~combined.index.duplicated(keep="first")]



## Combine NN predictions with meta data

There is no prediction of CD4 for Chow 2023 because all the CD4 T cells label = 0.

In [6]:
common_ids = combined.index.intersection(adata.obs.index)
missing_in_obs = combined.index.difference(adata.obs.index)
missing_in_df = adata.obs.index.difference(combined.index)
print(f"  - Matching IDs: {len(common_ids)}")
print(f"  - In df but not in adata.obs: {len(missing_in_obs)}")
print(f"  - In adata.obs but not in df: {len(missing_in_df)}")



  - Matching IDs: 322563
  - In df but not in adata.obs: 0
  - In adata.obs but not in df: 68222


In [7]:
adata.obs = adata.obs.join(combined, how="left")
print(f"✅ Merged {combined.shape[0]} rows and {combined.shape[1]} columns into adata.obs.")


✅ Merged 322563 rows and 2 columns into adata.obs.


In [8]:
adata.obs.to_csv(f"/Users/wsun/research/CAT/data/{cell_type}_with_NN_predictions.tsv", sep="\t", index=True)


In [9]:
adata.obs

,study2,n_genes_by_counts,total_counts,TRB_cdr3,TRB_v_gene,TRA_cdr3,TRA_v_gene,clone,cell_id,study,cancer_type,Patient,Sample,Treatment,Tissue,study_specific_CR_per_cell,study_specific_CR_by_cluster,barcode,TRA_j_gene,TRB_j_gene,TRA_nUMI,TRB_nUMI,CD4_Caushi_Tfh2_66g,CD4_Lowery_neg_37g,CD4_Lowery_pos_40g,CD4_Oh_CXCL13_50g,CD4_ave_Hanada_pos_9g,CD4_ave_Hanada_neg_4g,study_clone_id,study_clone_number,pos_score_CD4,neg_score_CD4,cancer_reactive_per_cell,cancer_reactive,total_cells_patient,clone_number_per_patient,clone_number_total,clone_n_patient,clone_number_per_patient_median,clone_freq_per_patient,TRA_antigen,TRB_antigen,non_human_antigen,label,Prediction,True Label
AGTGGGATCTGACCTC.58-THCA-Zheng_2021,Zheng_2021,1346,3215.0,CA*MRGFIMATPSVR,TRBV30,CADYSGGGADGLTF,TRAV13-1,TRAV13-1_CADYSGGGADGLTF_TRBV30_CA*MRGFIMATPSVR,AGTGGGATCTGACCTC.58-THCA,Zheng_2021,THCA,Zheng2021_THCA-P20190118,THCA-P20190118,NaN,Normal,False,False,AGTGGGATCTGACCTC.58,TRAJ45,TRBJ1-2,3.0,7.0,-1.238036,0.230081,-0.194868,-1.074365,-0.291763,2.613042,NaN,NaN,-0.699758,1.421562,False,False,1653,1,1,1,1.0,0.000605,NaN,NaN,NaN,0,9.780438e-07,0.0
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024,Chen_2024,1029,2930.0,CAAAATNNNEQFF,TRBV10-3,CAGSNTDKLIF,TRAV29/DV5,TRAV29/DV5_CAGSNTDKLIF_TRBV10-3_CAAAATNNNEQFF,CRC20-B-I_AACCGCGTCTGATACG,Chen_2024,COAD,Chen2024_P20,P20-B-I,anti-PD-1,Blood,False,False,AACCGCGTCTGATACG-1,TRAJ34,TRBJ2-1,4.0,15.0,-1.192411,-1.809830,0.722192,-1.408235,-0.983316,-0.329319,NaN,NaN,-0.715442,-1.069575,False,False,5957,1,1,1,1.0,0.000168,NaN,NaN,NaN,0,1.637786e-07,0.0
CRC04-B-II_ATTTCTGCAGTAAGCG-Chen_2024,Chen_2024,1236,3943.0,CAAAGAGTEAFF,TRBV6-5,CALSEARAGGSYIPTF,TRAV19,TRAV19_CALSEARAGGSYIPTF_TRBV6-5_CAAAGAGTEAFF,CRC04-B-II_ATTTCTGCAGTAAGCG,Chen_2024,COAD,Chen2024_P04,P04-B-II,anti-PD-1,Blood,False,False,ATTTCTGCAGTAAGCG-1,TRAJ6,TRBJ1-1,3.0,11.0,-0.978143,0.593070,-0.544647,-1.044290,-0.231579,0.355130,NaN,NaN,-0.699665,0.474100,False,False,16298,1,1,1,1.0,0.000061,NaN,NaN,NaN,0,6.141007e-08,0.0
CGCTTCAAGGCAAAGA-1_PEM15C5-Chow_2023,Chow_2023,971,2697.0,CAAAGGPKSGELFF,TRBV7-8,CVAEGHDMRF,TRAV12-1,TRAV12-1_CVAEGHDMRF_TRBV7-8_CAAAGGPKSGELFF,CGCTTCAAGGCAAAGA-1_PEM15C5,Chow_2023,EC,Chow2023_PEM15C5,GSM6514185,anti-PD-1,Blood,False,False,CGCTTCAAGGCAAAGA-1,TRAJ43,TRBJ2-2,1.0,11.0,-1.520697,-0.136410,-1.480302,-0.094554,-1.113605,1.870378,clonotype1406,NaN,-1.052290,0.866984,False,False,2058,1,1,1,1.0,0.000486,NaN,NaN,NaN,0,NaN,NaN
P47-TCAGCTCGTTCAGCGC-1-Liu_2025,Liu_2025,1041,1501.0,CAAAGQDNSPLHF,TRBV3-1,CAYRGGNSGGSNYKLTF,TRAV38-2/DV8,TRAV38-2/DV8_CAYRGGNSGGSNYKLTF_TRBV3-1_CAAAGQD...,P47-TCAGCTCGTTCAGCGC-1,Liu_2025,NSCLC,Liu2025_P47,P47,anti-PD-1,Tumor,False,False,1,TRAJ53,TRBJ1-6,NaN,NaN,-0.385449,-2.099794,-1.106830,-0.876728,-0.367543,-0.871389,P47_clonetype_2062,1.0,-0.684137,-1.485592,False,False,1880,1,1,1,1.0,0.000532,NaN,NaN,NaN,0,1.133308e-05,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GGGCATCCATACTCTT-1_PEM2C3-Chow_2023,Chow_2023,1108,2202.0,CWGTRPSISVPAVLAGGFTMSSSS,TRBV5-6,CAESLGASGGSYIPTF,TRAV5,TRAV5_CAESLGASGGSYIPTF_TRBV5-6_CWGTRPSISVPAVLA...,GGGCATCCATACTCTT-1_PEM2C3,Chow_2023,EC,Chow2023_PEM2C3,GSM6514151,anti-PD-1,Blood,True,False,GGGCATCCATACTCTT-1,TRAJ6,TRBJ2-1,3.0,3.0,-0.386322,-0.521064,-1.308884,-0.334458,-0.606441,-1.021454,clonotype1869,NaN,-0.659026,-0.771259,False,False,1669,1,1,1,1.0,0.000599,NaN,NaN,NaN,0,NaN,NaN
P14.ut.CCTAAAGAGATGTTAG-1-Liu_2022,Liu_2022,1233,3039.0,CWQLDQPQHF,TRBV19,CAVIVGGYGGSQGNLIF,TRAV12-2,TRAV12-2_CAVIVGGYGGSQGNLIF_TRBV19_CWQLDQPQHF,P14.ut.CCTAAAGAGATGTTAG-1,Liu_2022,NSCLC,Liu2022_P14,P14.pre,anti-PD-1,Tumor,False,False,CCTAAAGAGATGTTAG-1,TRAJ42,TRBJ1-5,4.0,10.0,0.438809,0.471930,0.906591,-0.258957,0.505871,0.157927,NaN,NaN,0.398078,0.314929,False,False,577,1,1,1,1.0,0.001733,NaN,NaN,NaN,0,6.190493e-05,0.0
GATCGATTCCCTCTTT.7-ESCA-Zheng_2021,Zheng_2021,1187,3256.0,C